# Stage S11 — Full Controlled Multimodal Training & Ablation Pipeline (`EXP0001–EXP0020`, `Blocks A–F`)
**Project:** Explainable Multimodal Cardiovascular Disease Risk Prediction (`STEP-029` & `STEP-031`)  
**Gate Lock:** `G02–G09` **PASS**. **`REL0001 Test` partition remains strictly SEALED until Gate `G11`.**

### 📌 Kaggle Inputs to Attach (`+ Add Input`):
1. **`REL0001_kaggle_train_val.zip`** (`Train = 66,338` + `Val = 12,724` tabular features & masks)
2. **`REL0001_ds0002_png_train_val.zip`** (`DS0002 CARDIO-ML-ED` Stage-A rotated `768×384` Train+Val ECG scans)
3. **`ptb-xl`** (Search Kaggle Public Datasets `+ Add Input -> ptb-xl` for `DS0001` 12-lead `.dat` waveforms; **0 MB upload!**)

### 📋 এই নোটবুক যে ২০টি এক্সপেরিমেন্ট চালাবে:
- **Block A (`EXP0001–EXP0010`):** Tier 1 Classical (`LR, RF, SVM, XGBoost, LightGBM`) + Neural (`TabularMLP`) + Unimodal Deep ECG (`CompactResNet1D`, `Conv1DBiLSTM`, `CompactCNN2D_PNG`, `ECGPatchTransformer`)
- **Block B (`EXP0011`):** Tier 2 WHO / Framingham Contextual Risk Score
- **Block C (`EXP0012–EXP0015`):** Tier 3 Early Concat (`EXP0012`), Late OOF Stacking (`EXP0013`), Gated Attention (`EXP0014`) vs Proposed **`CVDMMTransformer` (`EXP0015`)**
- **Block D (`EXP0016–EXP0017`):** Progressive Modality Gain (`C -> C+L -> C+L+S`, `C+L -> C+E+L`) & LOMO Ablation (`-C, -E, -L, -S`)
- **Block E (`EXP0018–EXP0019`):** Masking Ablation (`zero_fill` vs `~ModalityMask_CELS`) & PROBAST Sensitivity (`Prior_CABG`, `Family History`)
- **Block F (`EXP0020`):** `5`-Seed Stability (`[20261127..20261201]`) & Inference Missing-Modality Stress (`10–100%` masking + `DS0002 [MISSING_ECG]`)

In [ ]:
# Cell 1: Determinism, Hardware Lock, Kaggle Input Auto-Discovery & Stage-A Signal Caching (Train+Val Only)
import os, re, json, math, time, random, hashlib, zipfile, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.signal as sps
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import IsotonicRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, brier_score_loss,
    matthews_corrcoef, balanced_accuracy_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

TEST_SEALED = True
CANONICAL_SEEDS = [20261127, 20261128, 20261129, 20261130, 20261201]
PRIMARY_SEED = CANONICAL_SEEDS[0]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = Path('/kaggle/working/s11_outputs') if Path('/kaggle').exists() else Path('08_Checkpoints/binaries/s11_outputs')
CKPT_DIR = OUT_DIR / 'checkpoints'
PRED_DIR = OUT_DIR / 'val_predictions'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int = PRIMARY_SEED):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(PRIMARY_SEED)

# Auto-locate REL0001 train/val, DS0002 PNG folder & PTB-XL WFDB roots cleanly
SEARCH_ROOTS = [p for p in [Path('/kaggle/input'), Path('05_Preprocessing_Alignment/processed'), Path('Dataset'), Path('.')] if p.exists()]
TRAIN_SHARD = VAL_SHARD = PNG_DIR = PTB_ROOT = None
for r in SEARCH_ROOTS:
    if TRAIN_SHARD is None:
        hits = list(r.rglob('rel0001_train.jsonl'))
        if hits: TRAIN_SHARD = hits[0]
    if VAL_SHARD is None:
        hits = list(r.rglob('rel0001_val.jsonl'))
        if hits: VAL_SHARD = hits[0]
    if PNG_DIR is None:
        hits = list(r.rglob('ds0002_png_train_val_manifest.json'))
        if hits: PNG_DIR = hits[0].parent / 'ds0002_png'
        else:
            hits2 = list(r.rglob('ECGS'))
            if hits2: PNG_DIR = hits2[0]
    if PTB_ROOT is None:
        hits = list(r.rglob('records100'))
        if hits: PTB_ROOT = hits[0].parent
        else:
            hits5 = list(r.rglob('records500'))
            if hits5: PTB_ROOT = hits5[0].parent

assert TRAIN_SHARD is not None and VAL_SHARD is not None, 'Attach REL0001_kaggle_train_val.zip!'
print(f'Device={DEVICE} | PyTorch={torch.__version__} | TEST_SEALED={TEST_SEALED}')
print(f'TRAIN_SHARD={TRAIN_SHARD}')
print(f'VAL_SHARD  ={VAL_SHARD}')
print(f'PNG_DIR    ={PNG_DIR} | PTB_ROOT={PTB_ROOT}')

In [ ]:
# Cell 2: Load REL0001 Train/Val + Stage-A Vectorized WFDB & Red-Grid PNG Caching + Metrics/Calibrators
COHORT_MODALITY_SLICES = {
    'DS0001': {'clinical': (0,1,2,3,4,5,6,7,8,9,10), 'laboratory': (), 'lifestyle': ()},
    'DS0002': {'clinical': (0,1,2,3,4,5,6,7,12,13,14), 'laboratory': (8,9,10,11), 'lifestyle': ()},
    'DS0003': {'clinical': (0,1,2,3,4,5,6,7,8,9,10,11,24,25,26), 'laboratory': (12,13,14,15,16,17,18,19), 'lifestyle': (20,21,22,23,27,28,29,30)},
    'DS0004': {'clinical': (0,1,2,3,4,5,6,7,8,9,10,11,22,23,24), 'laboratory': (12,13,14,15), 'lifestyle': (16,17,18,19,20,21)},
}
COHORT_MODALITY_DIMS = {ds: {m: len(ix) for m, ix in mods.items()} for ds, mods in COHORT_MODALITY_SLICES.items()}

def load_jsonl_partition(part: str):
    if part == 'test' and TEST_SEALED:
        raise PermissionError('TEST_PARTITION_SEALED until Gate G11 PASS!')
    p = TRAIN_SHARD if part == 'train' else VAL_SHARD
    with open(p, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_recs = load_jsonl_partition('train')
val_recs = load_jsonl_partition('val')
print(f'Loaded REL0001 strictly Train/Val | Train={len(train_recs):,} | Val={len(val_recs):,}')

# Stage-A Vectorized WFDB Cache (100 Hz float16 (12, 1000) with 50 Hz notch + <=0.5 Hz Butterworth HP)
B_NOTCH_500, A_NOTCH_500 = sps.iirnotch(50.0, Q=30.0, fs=500.0)
SOS_HP_500 = sps.butter(2, 0.5, btype='highpass', fs=500.0, output='sos')
SOS_HP_100 = sps.butter(2, 0.5, btype='highpass', fs=100.0, output='sos')

def load_wfdb_12lead_100hz(ecg_ref: dict) -> np.ndarray | None:
    if PTB_ROOT is None or not ecg_ref:
        return None
    # Try 500 Hz first (true 50 Hz notch + 0.5 Hz HP -> 5:1 decimate), fallback to 100 Hz (0.5 Hz HP)
    hr_rel = ecg_ref.get('wfdb_hr_relpath', '')
    lr_rel = ecg_ref.get('wfdb_lr_relpath', '')
    hr_sub = hr_rel.split('records500/')[-1] if 'records500/' in hr_rel else ''
    lr_sub = lr_rel.split('records100/')[-1] if 'records100/' in lr_rel else ''
    hr_p = (PTB_ROOT / 'records500' / f'{hr_sub}.dat') if hr_sub else None
    lr_p = (PTB_ROOT / 'records100' / f'{lr_sub}.dat') if lr_sub else None
    if hr_p is not None and hr_p.is_file():
        arr = np.fromfile(hr_p, dtype='<i2').reshape(-1, 12).T.astype(np.float32) / 1000.0
        arr = sps.filtfilt(B_NOTCH_500, A_NOTCH_500, arr, axis=-1)
        arr = sps.sosfiltfilt(SOS_HP_500, arr, axis=-1)[:, ::5]
    elif lr_p is not None and lr_p.is_file():
        arr = np.fromfile(lr_p, dtype='<i2').reshape(-1, 12).T.astype(np.float32) / 1000.0
        arr = sps.sosfiltfilt(SOS_HP_100, arr, axis=-1)
    else:
        return None
    # Fix exact length=1000 and per-lead standardize safely inside [-10, 10]
    if arr.shape[1] >= 1000: arr = arr[:, :1000]
    else: arr = np.pad(arr, ((0,0), (0, 1000 - arr.shape[1])))
    mu = arr.mean(axis=-1, keepdims=True)
    sd = np.maximum(arr.std(axis=-1, keepdims=True), 1e-4)
    return np.clip((arr - mu) / sd, -10.0, 10.0).astype(np.float16)

def load_ds0002_clean_png(ecg_ref: dict, size_wh=(768, 384)) -> np.ndarray | None:
    if PNG_DIR is None or not ecg_ref or not ecg_ref.get('png_relpaths'):
        return None
    raw_name = Path(ecg_ref['png_relpaths'][0]).name
    p = PNG_DIR / raw_name
    if not p.is_file():
        rp = Path(raw_name)
        clean_stem = re.sub(r'\s+', '_', re.sub(r'\((\d+)\)$', r'-\1', rp.stem.strip()))
        clean_name = f'{clean_stem}{rp.suffix.strip().lower()}'
        p = PNG_DIR / clean_name
    if not p.is_file():
        return None
    img = Image.open(p)
    if img.width < img.height:
        img = img.rotate(90, expand=True)
    if (img.width, img.height) != size_wh:
        img = img.resize(size_wh, Image.Resampling.BILINEAR)
    rgb = np.asarray(img.convert('RGB'), dtype=np.float32)
    r, g, b = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
    # Skip double thresholding if Stage-A red-grid cleaning was already applied offline (R == G == B)
    if np.allclose(r, g) and np.allclose(g, b):
        gray = r
    else:
        cmax = np.maximum(np.maximum(r, g), b)
        red_ex = np.maximum(0.0, r - np.maximum(g, b))
        gray = np.minimum(255.0, 0.299*r + 0.587*g + 0.114*b + 0.75*red_ex)
        gray[(gray >= 155.0) & (cmax >= 165.0)] = 255.0
    return ((255.0 - gray) / 255.0)[None, ...].astype(np.float16)

t0 = time.time()
WFDB_CACHE = {}
PNG_CACHE = {}
for r in train_recs + val_recs:
    eid = r['EncounterID']
    if r['DS'] == 'DS0001' and PTB_ROOT is not None:
        sig = load_wfdb_12lead_100hz(r.get('ECGReference'))
        if sig is not None: WFDB_CACHE[eid] = sig
    elif r['DS'] == 'DS0002' and r['ModalityMask_CELS'][1] == 1 and PNG_DIR is not None:
        im = load_ds0002_clean_png(r.get('ECGReference'))
        if im is not None: PNG_CACHE[eid] = im
print(f'Cached in {time.time()-t0:.1f}s | WFDB={len(WFDB_CACHE):,} (of 19,634) | PNG={len(PNG_CACHE):,} (of 281)')

def compute_all_metrics(y_true, y_prob, thr: float = 0.5):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.clip(np.asarray(y_prob, dtype=np.float64), 1e-7, 1.0 - 1e-7)
    y_pred = (y_prob >= thr).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    order = np.argsort(y_prob)
    ece_q = sum((len(b)/len(y_true)) * abs(y_true[b].mean() - y_prob[b].mean()) for b in np.array_split(order, min(15, len(order))) if len(b))
    brier = float(brier_score_loss(y_true, y_prob))
    prev = float(y_true.mean())
    bss = 1.0 - brier / max(1e-9, prev * (1.0 - prev))
    return {
        'accuracy': round(float(accuracy_score(y_true, y_pred)), 6),
        'precision': round(float(precision_score(y_true, y_pred, zero_division=0)), 6),
        'recall': round(float(recall_score(y_true, y_pred, zero_division=0)), 6),
        'f1': round(float(f1_score(y_true, y_pred, zero_division=0)), 6),
        'roc_auc': round(float(roc_auc_score(y_true, y_prob)), 6),
        'pr_auc': round(float(average_precision_score(y_true, y_prob)), 6),
        'specificity': round(float(tn / max(1, tn + fp)), 6),
        'npv': round(float(tn / max(1, tn + fn)), 6),
        'balanced_accuracy': round(float(balanced_accuracy_score(y_true, y_pred)), 6),
        'mcc': round(float(matthews_corrcoef(y_true, y_pred)), 6),
        'brier': round(brier, 6),
        'bss': round(float(bss), 6),
        'ece_q15': round(float(ece_q), 6),
        'cm': [[int(tn), int(fp)], [int(fn), int(tp)]],
    }

def fit_temp_and_thresholds(y_val, raw_probs):
    y_val = np.asarray(y_val, dtype=np.int64)
    p_clip = np.clip(np.asarray(raw_probs, dtype=np.float64), 1e-6, 1.0 - 1e-6)
    logits = np.log(p_clip / (1.0 - p_clip))
    best_t, best_nll = 1.0, 1e18
    for t in np.linspace(0.25, 3.0, 111):
        pc = np.clip(1.0 / (1.0 + np.exp(-np.clip(logits / t, -50, 50))), 1e-7, 1 - 1e-7)
        nll = -(y_val * np.log(pc) + (1 - y_val) * np.log(1 - pc)).mean()
        if nll < best_nll: best_nll, best_t = nll, float(t)
    cal_p = 1.0 / (1.0 + np.exp(-np.clip(logits / best_t, -50, 50)))
    best_j, tau_j, best_spec, tau_s85 = -1.0, 0.5, -1.0, 0.1
    for tau in np.linspace(0.05, 0.95, 181):
        pred = cal_p >= tau
        sens = ((y_val == 1) & pred).sum() / max(1, (y_val == 1).sum())
        spec = ((y_val == 0) & (~pred)).sum() / max(1, (y_val == 0).sum())
        if (sens + spec - 1.0) > best_j: best_j, tau_j = float(sens + spec - 1.0), float(tau)
        if sens >= 0.85 and spec > best_spec: best_spec, tau_s85 = float(spec), float(tau)
    return cal_p, round(best_t, 4), round(tau_j, 4), round(tau_s85, 4)

RESULTS_LOG = []
def log_experiment(exp_id, block, tier, mc_id, cohort, y_val, raw_probs, seed=PRIMARY_SEED, notes=''):
    cal_p, temp, tau_j, tau_s85 = fit_temp_and_thresholds(y_val, raw_probs)
    m_raw = compute_all_metrics(y_val, raw_probs, thr=0.5)
    m_cal = compute_all_metrics(y_val, cal_p, thr=tau_j)
    row = {
        'EXP': exp_id, 'Block': block, 'Tier': tier, 'MC': mc_id, 'Cohort': cohort, 'Seed': seed,
        'Val_ROCAUC_Raw': m_raw['roc_auc'], 'Val_PRAUC': m_cal['pr_auc'], 'Val_Brier_Raw': m_raw['brier'],
        'Val_Brier_Cal': m_cal['brier'], 'Val_ECE_Cal': m_cal['ece_q15'], 'Val_F1_Youden': m_cal['f1'],
        'Val_Acc_Youden': m_cal['accuracy'], 'Temp': temp, 'Tau_Youden': tau_j, 'Tau_Sens85': tau_s85,
        'CM_Youden': json.dumps(m_cal['cm']), 'Notes': notes
    }
    RESULTS_LOG.append(row)
    np.savez_compressed(PRED_DIR / f'{exp_id}_{mc_id}_{cohort}_seed{seed}_val.npz', y_val=np.asarray(y_val), raw_prob=np.asarray(raw_probs), cal_prob=cal_p)
    return row

In [ ]:
# Cell 3: Blocks A & B — Tier 1 Tabular (EXP0001-EXP0006) & Tier 2 Clinical Score (EXP0011)
class TabularUnimodalNet(nn.Module):
    def __init__(self, d_in: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64, 32), nn.GELU(), nn.Linear(32, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

TAB_VAL_PROBS = {}   # (ds_id, mc_id) -> np.ndarray val probabilities
TAB_OOF_PROBS = {}   # (ds_id, mc_id) -> 5-fold Train OOF probabilities for EXP0013 stacking

def fit_tabular_mlp(X_tr, y_tr, X_va, y_va, pos_w, epochs=25, seed=PRIMARY_SEED):
    seed_everything(seed)
    m = TabularUnimodalNet(X_tr.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w], device=DEVICE))
    Xt, yt = torch.from_numpy(X_tr).float(), torch.from_numpy(y_tr).float()
    Xv = torch.from_numpy(X_va).float().to(DEVICE)
    best_auc, best_p = -1.0, None
    bs = min(256, max(32, len(X_tr)//4))
    for _ in range(epochs):
        m.train()
        perm = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), bs):
            idx = perm[i:i+bs]
            opt.zero_grad(set_to_none=True)
            loss = crit(m(Xt[idx].to(DEVICE)), yt[idx].to(DEVICE))
            loss.backward(); opt.step()
        sched.step()
        m.eval()
        with torch.no_grad():
            vp = torch.sigmoid(m(Xv)).cpu().numpy()
            auc = roc_auc_score(y_va, vp)
            if auc > best_auc: best_auc, best_p = auc, vp
    return best_p

for ds_id in ['DS0001', 'DS0002', 'DS0003', 'DS0004']:
    tr_sub = [r for r in train_recs if r['DS'] == ds_id]
    va_sub = [r for r in val_recs if r['DS'] == ds_id]
    X_tr = np.asarray([r['TabularFeatures'] for r in tr_sub], dtype=np.float32)
    y_tr = np.asarray([r['Target_y'] for r in tr_sub], dtype=np.int64)
    X_va = np.asarray([r['TabularFeatures'] for r in va_sub], dtype=np.float32)
    y_va = np.asarray([r['Target_y'] for r in va_sub], dtype=np.int64)
    pos_w = float((y_tr == 0).sum()) / max(1.0, float((y_tr == 1).sum()))

    # Small Val grid (2 regularized vs deep presets) prevents depth=6 overfitting on DS0002 (390 rows) & DS0003 (1,170 rows)
    specs = [
        ('EXP0001', 'MC0001', [lambda: LogisticRegression(C=0.5, max_iter=1000, class_weight='balanced', random_state=PRIMARY_SEED), lambda: LogisticRegression(C=2.0, max_iter=1000, class_weight='balanced', random_state=PRIMARY_SEED)]),
        ('EXP0002', 'MC0002', [lambda: RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', n_jobs=-1, random_state=PRIMARY_SEED), lambda: RandomForestClassifier(n_estimators=300, max_depth=14, class_weight='balanced', n_jobs=-1, random_state=PRIMARY_SEED)]),
        ('EXP0004', 'MC0004', [lambda: XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, scale_pos_weight=pos_w, eval_metric='auc', random_state=PRIMARY_SEED, n_jobs=-1), lambda: XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, scale_pos_weight=pos_w, eval_metric='auc', random_state=PRIMARY_SEED, n_jobs=-1)]),
        ('EXP0005', 'MC0005', [lambda: LGBMClassifier(n_estimators=200, max_depth=4, num_leaves=15, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, scale_pos_weight=pos_w, random_state=PRIMARY_SEED, n_jobs=-1, verbose=-1), lambda: LGBMClassifier(n_estimators=400, num_leaves=31, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, scale_pos_weight=pos_w, random_state=PRIMARY_SEED, n_jobs=-1, verbose=-1)]),
    ]
    for exp_id, mc_id, builders in specs:
        best_clf, best_vp, best_auc_val = None, None, -1.0
        best_builder = builders[0]
        for bfn in builders:
            cand = bfn(); cand.fit(X_tr, y_tr)
            vp_c = cand.predict_proba(X_va)[:, 1]
            auc_c = roc_auc_score(y_va, vp_c)
            if auc_c > best_auc_val:
                best_auc_val, best_clf, best_vp, best_builder = auc_c, cand, vp_c, bfn
        TAB_VAL_PROBS[(ds_id, mc_id)] = best_vp
        log_experiment(exp_id, 'Block_A', 'unimodal', mc_id, ds_id, y_va, best_vp)
        if mc_id == 'MC0004':
            oof = np.zeros(len(y_tr), dtype=np.float32)
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=PRIMARY_SEED)
            for ti, vi in skf.split(X_tr, y_tr):
                m_f = best_builder(); m_f.fit(X_tr[ti], y_tr[ti])
                oof[vi] = m_f.predict_proba(X_tr[vi])[:, 1]
            TAB_OOF_PROBS[ds_id] = oof
            with open(CKPT_DIR / f'{exp_id}_{ds_id}_xgb.pkl', 'wb') as f: pickle.dump(best_clf, f)

    # EXP0003 SVM-RBF (deterministic subsample<=12K on DS0001/DS0004 for tractable RBF speed)
    if len(X_tr) > 12000:
        idx_svm = np.random.default_rng(PRIMARY_SEED).choice(len(X_tr), size=12000, replace=False)
        Xs, ys = X_tr[idx_svm], y_tr[idx_svm]
    else:
        Xs, ys = X_tr, y_tr
    svm = SVC(C=1.0, kernel='rbf', probability=True, class_weight='balanced', random_state=PRIMARY_SEED).fit(Xs, ys)
    log_experiment('EXP0003', 'Block_A', 'unimodal', 'MC0003', ds_id, y_va, svm.predict_proba(X_va)[:, 1])

    # EXP0006 TabularMLP
    mlp_p = fit_tabular_mlp(X_tr, y_tr, X_va, y_va, pos_w)
    log_experiment('EXP0006', 'Block_A', 'unimodal', 'MC0006', ds_id, y_va, mlp_p)

    # EXP0011 Block B WHO/Framingham Clinical Score Context (DS0003/DS0004)
    if ds_id in ('DS0003', 'DS0004'):
        z_who = -0.35 + 0.55 * X_va[:, 0] + 0.35 * X_va[:, 2] + 0.25 * X_va[:, 12] + 0.20 * X_va[:, 14]
        p_who = 1.0 / (1.0 + np.exp(-np.clip(z_who, -50, 50)))
        log_experiment('EXP0011', 'Block_B', 'clinical_score', 'MC0011', ds_id, y_va, p_who)

print(pd.DataFrame(RESULTS_LOG)[['EXP', 'MC', 'Cohort', 'Val_ROCAUC_Raw', 'Val_F1_Youden', 'Val_ECE_Cal']].to_string(index=False))

In [ ]:
# Cell 4: Block A Unimodal Deep ECG Encoders (EXP0007-EXP0010) & Train-Only ECG Embeddings
class SEBlock1D(nn.Module):
    def __init__(self, c: int):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(c, max(4, c//4)), nn.SiLU(), nn.Linear(max(4, c//4), c), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x.mean(dim=-1)).unsqueeze(-1)

class ResBlock1D(nn.Module):
    def __init__(self, ci: int, co: int, stride: int = 2, drop: float = 0.1):
        super().__init__()
        self.c1 = nn.Conv1d(ci, co, 5, stride=stride, padding=2, bias=False); self.b1 = nn.BatchNorm1d(co)
        self.c2 = nn.Conv1d(co, co, 5, stride=1, padding=2, bias=False); self.b2 = nn.BatchNorm1d(co)
        self.se = SEBlock1D(co); self.drop = nn.Dropout(drop)
        self.sh = nn.Sequential(nn.Conv1d(ci, co, 1, stride=stride, bias=False), nn.BatchNorm1d(co)) if (stride!=1 or ci!=co) else nn.Identity()
    def forward(self, x): return F.gelu(self.se(self.b2(self.c2(self.drop(F.gelu(self.b1(self.c1(x))))))) + self.sh(x))

class CompactResNet1D(nn.Module):
    def __init__(self, embed_dim: int = 64):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(12, 32, 7, stride=2, padding=3, bias=False), nn.BatchNorm1d(32), nn.GELU())
        self.blocks = nn.Sequential(ResBlock1D(32, 32, 2), ResBlock1D(32, 64, 2), ResBlock1D(64, embed_dim, 2))
        self.head = nn.Linear(embed_dim, 1)
    def encode(self, x): return self.blocks(self.stem(x)).mean(dim=-1)
    def forward(self, x): return self.head(self.encode(x)).squeeze(-1)

class Conv1DBiLSTMEncoder(nn.Module):
    def __init__(self, embed_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(12, 32, 7, stride=2, padding=3, bias=False), nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2), nn.Conv1d(32, 48, 5, stride=2, padding=2, bias=False), nn.BatchNorm1d(48), nn.GELU())
        self.lstm = nn.LSTM(48, embed_dim//2, batch_first=True, bidirectional=True)
        self.head = nn.Linear(embed_dim, 1)
    def encode(self, x):
        out, _ = self.lstm(self.conv(x).transpose(1, 2)); return out.mean(dim=1)
    def forward(self, x): return self.head(self.encode(x)).squeeze(-1)

class CompactCNN2DPNG(nn.Module):
    def __init__(self, embed_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, 2, 1, bias=False), nn.BatchNorm2d(16), nn.GELU(),
            nn.Conv2d(16, 32, 3, 2, 1, bias=False), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 48, 3, 2, 1, bias=False), nn.BatchNorm2d(48), nn.GELU(),
            nn.Conv2d(48, embed_dim, 3, 2, 1, bias=False), nn.BatchNorm2d(embed_dim), nn.GELU()
        )
        # Dual Max+Avg pooling preserves sparse bright QRS trace activations over white paper
        self.proj = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.LayerNorm(embed_dim), nn.GELU(), nn.Dropout(0.3))
        self.head = nn.Linear(embed_dim, 1)
    def encode(self, img):
        h = self.conv(img)
        pooled = torch.cat([F.adaptive_max_pool2d(h, 1).flatten(1), F.adaptive_avg_pool2d(h, 1).flatten(1)], dim=1)
        return self.proj(pooled)
    def forward(self, img): return self.head(self.encode(img)).squeeze(-1)

class ECGPatchTransformer(nn.Module):
    def __init__(self, d_model: int = 64):
        super().__init__()
        self.patch = nn.Conv1d(12, d_model, kernel_size=25, stride=25)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        enc = nn.TransformerEncoderLayer(d_model, nhead=4, dim_feedforward=128, dropout=0.1, activation='gelu', batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, num_layers=2)
        self.head = nn.Linear(d_model, 1)
    def encode(self, x):
        p = self.patch(x).transpose(1, 2)
        return self.enc(torch.cat([self.cls.expand(p.size(0), -1, -1), p], dim=1))[:, 0]
    def forward(self, x): return self.head(self.encode(x)).squeeze(-1)

def train_unimodal_ecg(model_cls, exp_id, mc_id, cache_dict, cohort, epochs=25, bs=64):
    tr_items = [r for r in train_recs if r['DS'] == cohort and r['EncounterID'] in cache_dict]
    va_items = [r for r in val_recs if r['DS'] == cohort and r['EncounterID'] in cache_dict]
    if len(tr_items) < 20 or len(va_items) < 10:
        print(f'Skipping {exp_id} ({mc_id}): ECG cache absent for {cohort}!'); return None, {}, {}
    seed_everything(PRIMARY_SEED)
    Xtr = np.stack([cache_dict[r['EncounterID']] for r in tr_items])
    ytr = np.asarray([r['Target_y'] for r in tr_items], dtype=np.float32)
    Xva = np.stack([cache_dict[r['EncounterID']] for r in va_items])
    yva = np.asarray([r['Target_y'] for r in va_items], dtype=np.int64)
    pos_w = float((ytr == 0).sum()) / max(1.0, float((ytr == 1).sum()))
    model = model_cls().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_w], device=DEVICE))
    best_auc, best_p, best_state, wait = -1.0, None, None, 0
    for _ in range(epochs):
        model.train(); perm = np.random.permutation(len(Xtr))
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i+bs]
            if len(idx) < 2: continue
            opt.zero_grad(set_to_none=True)
            bx = torch.from_numpy(Xtr[idx].astype(np.float32)).to(DEVICE)
            by = torch.from_numpy(ytr[idx]).to(DEVICE)
            loss = crit(model(bx), by); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sched.step(); model.eval()
        vp_parts = []
        with torch.no_grad():
            for i in range(0, len(Xva), bs):
                bx = torch.from_numpy(Xva[i:i+bs].astype(np.float32)).to(DEVICE)
                vp_parts.append(torch.sigmoid(model(bx)).cpu().numpy())
        vp = np.concatenate(vp_parts)
        auc = roc_auc_score(yva, vp)
        if auc > best_auc:
            best_auc, best_p, wait = auc, vp, 0
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= 7: break
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()}); model.eval()
    torch.save({'exp_id': exp_id, 'mc_id': mc_id, 'val_roc_auc': best_auc, 'state_dict': best_state}, CKPT_DIR / f'{exp_id}_{mc_id}_best_val.pt')
    log_experiment(exp_id, 'Block_A', 'unimodal', mc_id, cohort, yva, best_p)
    # Extract frozen 64-D embeddings (Train+Val only) for EXP0012 EarlyConcat
    emb_map, prob_map = {}, {}
    with torch.no_grad():
        for subset, Xsub in [(tr_items, Xtr), (va_items, Xva)]:
            for i in range(0, len(Xsub), bs):
                bx = torch.from_numpy(Xsub[i:i+bs].astype(np.float32)).to(DEVICE)
                eb = model.encode(bx).cpu().numpy(); pr = torch.sigmoid(model.head(torch.from_numpy(eb).to(DEVICE)).squeeze(-1)).cpu().numpy()
                for j, rec in enumerate(subset[i:i+bs]):
                    emb_map[rec['EncounterID']] = eb[j]; prob_map[rec['EncounterID']] = float(pr[j])
    return model, emb_map, prob_map

ECG_EMB, ECG_PROB = {}, {}
_, e1, p1 = train_unimodal_ecg(CompactResNet1D, 'EXP0007', 'MC0007', WFDB_CACHE, 'DS0001', epochs=25, bs=64)
ECG_EMB.update(e1); ECG_PROB.update(p1)
train_unimodal_ecg(Conv1DBiLSTMEncoder, 'EXP0008', 'MC0008', WFDB_CACHE, 'DS0001', epochs=25, bs=64)
_, e2, p2 = train_unimodal_ecg(CompactCNN2DPNG, 'EXP0009', 'MC0009', PNG_CACHE, 'DS0002', epochs=30, bs=32)
ECG_EMB.update(e2); ECG_PROB.update(p2)
train_unimodal_ecg(ECGPatchTransformer, 'EXP0010', 'MC0010', WFDB_CACHE, 'DS0001', epochs=25, bs=64)
print('Block A Deep ECG complete. Latest rows:')
print(pd.DataFrame(RESULTS_LOG[-4:])[['EXP', 'MC', 'Cohort', 'Val_ROCAUC_Raw', 'Val_F1_Youden']].to_string(index=False))

In [ ]:
# Cell 5: Block C — Tier 3 Simple Fusion (EXP0012-14) vs Tier 4 Proposed CVDMMTransformer (EXP0015)
def align_early_concat(recs, width=39):
    ds_map = {'DS0001': 0, 'DS0002': 1, 'DS0003': 2, 'DS0004': 3}
    X = np.zeros((len(recs), width + 64), dtype=np.float32)
    y = np.asarray([r['Target_y'] for r in recs], dtype=np.int64)
    for i, r in enumerate(recs):
        X[i, :4] = r['ModalityMask_CELS']; X[i, 4 + ds_map[r['DS']]] = 1.0
        tf = r['TabularFeatures'][:width-8]; X[i, 8:8+len(tf)] = tf
        if r['EncounterID'] in ECG_EMB: X[i, width:] = ECG_EMB[r['EncounterID']]
    return X, y

Xtr_ec, ytr_ec = align_early_concat(train_recs); Xva_ec, yva_ec = align_early_concat(val_recs)
pw_all = float((ytr_ec == 0).sum()) / max(1.0, float((ytr_ec == 1).sum()))
xgb_ec = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=pw_all, eval_metric='auc', random_state=PRIMARY_SEED, n_jobs=-1)
xgb_ec.fit(Xtr_ec, ytr_ec)
pva_ec = xgb_ec.predict_proba(Xva_ec)[:, 1]
log_experiment('EXP0012', 'Block_C', 'simple_fusion', 'MC0012', 'ALL_COHORTS', yva_ec, pva_ec)

# EXP0013 Late OOF Logit Stacker (MC0013)
def build_stack_feats(recs, use_oof=True):
    Fmat = np.zeros((len(recs), 6), dtype=np.float32)
    va_offset = {ds: 0 for ds in ['DS0001','DS0002','DS0003','DS0004']}
    tr_offset = {ds: 0 for ds in ['DS0001','DS0002','DS0003','DS0004']}
    for i, r in enumerate(recs):
        ds = r['DS']
        if use_oof:
            ptab = TAB_OOF_PROBS[ds][tr_offset[ds]]; tr_offset[ds] += 1
        else:
            ptab = TAB_VAL_PROBS[(ds, 'MC0004')][va_offset[ds]]; va_offset[ds] += 1
        pecg = ECG_PROB.get(r['EncounterID'], 0.5)
        Fmat[i, 0] = math.log(min(max(ptab, 1e-6), 1-1e-6) / (1 - min(max(ptab, 1e-6), 1-1e-6)))
        Fmat[i, 1] = math.log(min(max(pecg, 1e-6), 1-1e-6) / (1 - min(max(pecg, 1e-6), 1-1e-6))) if r['EncounterID'] in ECG_PROB else 0.0
        Fmat[i, 2:] = r['ModalityMask_CELS']
    return Fmat
stack_clf = LogisticRegression(C=1.0, class_weight='balanced', random_state=PRIMARY_SEED).fit(build_stack_feats(train_recs, True), ytr_ec)
log_experiment('EXP0013', 'Block_C', 'simple_fusion', 'MC0013', 'ALL_COHORTS', yva_ec, stack_clf.predict_proba(build_stack_feats(val_recs, False))[:, 1])

# Neural Fusion Architectures (MC0014 GatedAttention & MC0015 Proposed CVDMMTransformer)
class TabularAdapter(nn.Module):
    def __init__(self, d_in: int, d_model: int = 64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(0.15), nn.Linear(d_model, d_model))
    def forward(self, x): return self.net(x)

class CVDMMTransformerUnified(nn.Module):
    def __init__(self, d_model: int = 64, nhead: int = 4, n_layers: int = 2, mod_drop_p: float = 0.15, use_key_mask: bool = True, gated_only: bool = False):
        super().__init__()
        self.d_model = d_model; self.mod_drop_p = mod_drop_p; self.use_key_mask = use_key_mask; self.gated_only = gated_only
        self.tab_adapters = nn.ModuleDict({f'{ds}__{m}': TabularAdapter(dim, d_model) for ds, mods in COHORT_MODALITY_DIMS.items() for m, dim in mods.items() if dim > 0})
        self.ecg_1d = CompactResNet1D(d_model)
        self.ecg_2d = CompactCNN2DPNG(d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.mod_type = nn.Parameter(torch.randn(1, 4, d_model) * 0.02)
        if gated_only:
            self.gate = nn.Sequential(nn.Linear(d_model, 32), nn.Tanh(), nn.Linear(32, 1))
        else:
            enc = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=128, dropout=0.1, activation='gelu', batch_first=True, norm_first=True)
            self.encoder = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, 32), nn.GELU(), nn.Dropout(0.1), nn.Linear(32, 1))

    def forward(self, ds_id: str, tab_mods: dict, mask_cels: torch.Tensor, ecg_1d=None, ecg_2d=None):
        b, dev = mask_cels.size(0), mask_cels.device
        eff_mask = mask_cels.bool()
        if self.training and self.mod_drop_p > 0.0:
            drop = torch.rand_like(eff_mask.float()) < self.mod_drop_p
            kept = eff_mask & (~drop); all_zero = kept.sum(dim=1) == 0; kept[all_zero] = eff_mask[all_zero]; eff_mask = kept
        toks = []
        for idx, mod in enumerate(['clinical', 'ecg', 'laboratory', 'lifestyle']):
            if mod == 'ecg':
                if ecg_1d is not None: emb = self.ecg_1d.encode(ecg_1d) * eff_mask[:, 1:2].float()
                elif ecg_2d is not None: emb = self.ecg_2d.encode(ecg_2d) * eff_mask[:, 1:2].float()
                else: emb = torch.zeros(b, self.d_model, device=dev)
            else:
                key = f'{ds_id}__{mod}'
                emb = self.tab_adapters[key](tab_mods[mod]) * eff_mask[:, idx:idx+1].float() if (key in self.tab_adapters and mod in tab_mods) else torch.zeros(b, self.d_model, device=dev)
            toks.append(emb + self.mod_type[:, idx, :])
        stacked = torch.stack(toks, dim=1)
        if self.gated_only:
            sc = self.gate(stacked).squeeze(-1).masked_fill(~eff_mask, -1e9)
            w = F.softmax(sc, dim=-1).unsqueeze(-1)
            return self.head(self.norm((stacked * w).sum(dim=1))).squeeze(-1)
        seq = torch.cat([self.cls_token.expand(b, -1, -1), stacked], dim=1)
        pad_mask = torch.cat([torch.zeros(b, 1, dtype=torch.bool, device=dev), ~eff_mask], dim=1) if self.use_key_mask else None
        out = self.norm(self.encoder(seq, src_key_padding_mask=pad_mask))
        return self.head(out[:, 0]).squeeze(-1)

def prepare_cohort_tensors(recs):
    by_ds = {}
    for ds_id in ['DS0001', 'DS0002', 'DS0003', 'DS0004']:
        sub = [r for r in recs if r['DS'] == ds_id]
        X = np.asarray([r['TabularFeatures'] for r in sub], dtype=np.float32)
        mods = {m: torch.from_numpy(X[:, list(ix)]) for m, ix in COHORT_MODALITY_SLICES[ds_id].items() if len(ix) > 0}
        masks = torch.tensor([r['ModalityMask_CELS'] for r in sub], dtype=torch.bool)
        y = torch.tensor([r['Target_y'] for r in sub], dtype=torch.float32)
        eids = [r['EncounterID'] for r in sub]
        ecg1d = np.stack([WFDB_CACHE[e] if e in WFDB_CACHE else np.zeros((12, 1000), dtype=np.float16) for e in eids]) if ds_id == 'DS0001' else None
        ecg2d = np.stack([PNG_CACHE[e] if e in PNG_CACHE else np.zeros((1, 384, 768), dtype=np.float16) for e in eids]) if ds_id == 'DS0002' else None
        by_ds[ds_id] = {'mods': mods, 'mask': masks, 'y': y, 'ecg1d': ecg1d, 'ecg2d': ecg2d, 'n': len(sub)}
    return by_ds

TR_COHORTS = prepare_cohort_tensors(train_recs)
VA_COHORTS = prepare_cohort_tensors(val_recs)

def eval_multicohort_model(model, coh_dict, override_mask=None):
    model.eval(); ys, ps, per_ds = [], [], {}
    with torch.no_grad():
        for ds_id, d in coh_dict.items():
            n, probs_ds = d['n'], []
            bs = 64 if ds_id in ('DS0001', 'DS0002') else 512
            for i in range(0, n, bs):
                bm = {m: t[i:i+bs].to(DEVICE) for m, t in d['mods'].items()}
                bmask = (override_mask(ds_id, d['mask'][i:i+bs]) if override_mask else d['mask'][i:i+bs]).to(DEVICE)
                e1 = torch.from_numpy(d['ecg1d'][i:i+bs].astype(np.float32)).to(DEVICE) if d['ecg1d'] is not None else None
                e2 = torch.from_numpy(d['ecg2d'][i:i+bs].astype(np.float32)).to(DEVICE) if d['ecg2d'] is not None else None
                probs_ds.append(torch.sigmoid(model(ds_id, bm, bmask, ecg_1d=e1, ecg_2d=e2)).cpu().numpy())
            p_ds = np.concatenate(probs_ds); y_ds = d['y'].numpy().astype(np.int64)
            per_ds[ds_id] = (y_ds, p_ds); ys.append(y_ds); ps.append(p_ds)
    return np.concatenate(ys), np.concatenate(ps), per_ds

def train_unified_fusion(exp_id, block, tier, mc_id, *, seed=PRIMARY_SEED, epochs=30, mod_drop_p=0.15, use_key_mask=True, gated_only=False):
    seed_everything(seed)
    model = CVDMMTransformerUnified(mod_drop_p=mod_drop_p, use_key_mask=use_key_mask, gated_only=gated_only).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
    pos_weights = {ds: torch.tensor([float((d['y']==0).sum().item())/max(1.0, float((d['y']==1).sum().item()))], device=DEVICE) for ds, d in TR_COHORTS.items()}
    best_macro_auc, best_state, best_eval, wait = -1.0, None, None, 0
    for ep in range(1, epochs + 1):
        model.train()
        # Interleave batches across all 4 cohorts + repeat smaller DS0002/DS0003 cohorts gently to avoid DS0004 domination
        epoch_batches = []
        bs_map = {'DS0001': 64, 'DS0002': 32, 'DS0003': 64, 'DS0004': 512}
        rep_map = {'DS0001': 1, 'DS0002': 4, 'DS0003': 2, 'DS0004': 1}
        for ds_id, d in TR_COHORTS.items():
            bs = bs_map[ds_id]
            for _ in range(rep_map[ds_id]):
                perm = torch.randperm(d['n'])
                for i in range(0, d['n'], bs):
                    idx = perm[i:i+bs]
                    if len(idx) >= 2:
                        epoch_batches.append((ds_id, idx))
        random.shuffle(epoch_batches)
        for ds_id, idx in epoch_batches:
            d = TR_COHORTS[ds_id]
            opt.zero_grad(set_to_none=True)
            bm = {m: t[idx].to(DEVICE) for m, t in d['mods'].items()}
            bmask = d['mask'][idx].to(DEVICE); by = d['y'][idx].to(DEVICE)
            e1 = torch.from_numpy(d['ecg1d'][idx.numpy()].astype(np.float32)).to(DEVICE) if d['ecg1d'] is not None else None
            e2 = torch.from_numpy(d['ecg2d'][idx.numpy()].astype(np.float32)).to(DEVICE) if d['ecg2d'] is not None else None
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                loss = F.binary_cross_entropy_with_logits(model(ds_id, bm, bmask, e1, e2), by, pos_weight=pos_weights[ds_id])
            scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
        sched.step()
        yv, pv, per_ds = eval_multicohort_model(model, VA_COHORTS)
        macro_auc = float(np.mean([roc_auc_score(yd, pd_) for yd, pd_ in per_ds.values()]))
        if macro_auc > best_macro_auc:
            best_macro_auc, wait, best_eval = macro_auc, 0, (yv, pv, per_ds)
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= 7: break
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()}); model.eval()
    torch.save({'exp_id': exp_id, 'mc_id': mc_id, 'seed': seed, 'val_macro_roc_auc': best_macro_auc, 'params': sum(p.numel() for p in model.parameters() if p.requires_grad), 'state_dict': best_state}, CKPT_DIR / f'{exp_id}_{mc_id}_seed{seed}_best_val.pt')
    yv, pv, per_ds = best_eval
    log_experiment(exp_id, block, tier, mc_id, 'ALL_COHORTS', yv, pv, seed=seed, notes=f'macro_auc={best_macro_auc:.4f};params={sum(p.numel() for p in model.parameters() if p.requires_grad)}')
    for ds_id, (yd, pd_) in per_ds.items():
        log_experiment(exp_id, block, tier, mc_id, ds_id, yd, pd_, seed=seed)
    return model

gated_model = train_unified_fusion('EXP0014', 'Block_C', 'simple_fusion', 'MC0014', epochs=30, gated_only=True)
champion_model = train_unified_fusion('EXP0015', 'Block_C', 'transformer_fusion', 'MC0015', epochs=30, gated_only=False)
print(pd.DataFrame(RESULTS_LOG)[pd.DataFrame(RESULTS_LOG)['EXP'].isin(['EXP0012','EXP0013','EXP0014','EXP0015'])][['EXP','MC','Cohort','Val_ROCAUC_Raw','Val_F1_Youden','Val_ECE_Cal']].to_string(index=False))

In [ ]:
# Cell 6: Blocks D, E & F — Progressive/LOMO Ablations (EXP0016-17), Masking/PROBAST (EXP0018-19) & 5-Seed + Missing Stress (EXP0020)
# EXP0016: Progressive Modality Gain C -> C+L -> C+L+S on DS0003 & DS0004 (XGBoost matched)
for ds_id in ('DS0003', 'DS0004'):
    tr_s = [r for r in train_recs if r['DS'] == ds_id]; va_s = [r for r in val_recs if r['DS'] == ds_id]
    Xtr_d = np.asarray([r['TabularFeatures'] for r in tr_s], dtype=np.float32); ytr_d = np.asarray([r['Target_y'] for r in tr_s])
    Xva_d = np.asarray([r['TabularFeatures'] for r in va_s], dtype=np.float32); yva_d = np.asarray([r['Target_y'] for r in va_s])
    pw = float((ytr_d == 0).sum()) / max(1.0, float((ytr_d == 1).sum()))
    sl = COHORT_MODALITY_SLICES[ds_id]
    for tag, cols in [('C_only', list(sl['clinical'])), ('C_plus_L', list(sl['clinical'])+list(sl['laboratory'])), ('C_plus_L_plus_S', list(sl['clinical'])+list(sl['laboratory'])+list(sl['lifestyle']))]:
        c = XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, scale_pos_weight=pw, eval_metric='auc', random_state=PRIMARY_SEED, n_jobs=-1)
        c.fit(Xtr_d[:, cols], ytr_d)
        log_experiment('EXP0016', 'Block_D', 'ablation', 'MC0004', f'{ds_id}_{tag}', yva_d, c.predict_proba(Xva_d[:, cols])[:, 1])

# EXP0017: LOMO Leave-One-Modality-Out Evaluation on Champion CVDMMTransformer
for drop_idx, drop_name in [(0, 'drop_Clinical'), (1, 'drop_ECG'), (2, 'drop_Lab'), (3, 'drop_Lifestyle')]:
    def _lomo(ds_id, m, idx=drop_idx):
        m2 = m.clone(); m2[:, idx] = False; return m2
    yv, pv, _ = eval_multicohort_model(champion_model, VA_COHORTS, override_mask=_lomo)
    log_experiment('EXP0017', 'Block_D', 'ablation', 'MC0015', f'ALL_LOMO_{drop_name}', yv, pv)

# EXP0018: Masking Mechanism Ablation (naive zero_fill without Transformer src_key_padding_mask)
train_unified_fusion('EXP0018', 'Block_E', 'ablation', 'MC0015', epochs=25, use_key_mask=False)

# EXP0019: PROBAST Structural Sensitivity on DS0002 (excluding Prior_CABG=1 rows if present + cohort eval)
no_cabg_idx = [i for i, r in enumerate([r for r in val_recs if r['DS']=='DS0002']) if (r.get('ECGReference') or {}).get('prior_cabg_flag', 0) == 0]
_, _, per_ds_champ = eval_multicohort_model(champion_model, VA_COHORTS)
y_ds2_all, p_ds2_all = per_ds_champ['DS0002']
log_experiment('EXP0019', 'Block_E', 'ablation', 'MC0015', 'DS0002_noCABG', y_ds2_all[no_cabg_idx], p_ds2_all[no_cabg_idx])

# EXP0020: Block F 5-Seed Variance + Missing-Modality Degradation (10-100% ECG/Lab/Life + DS0002 [MISSING_ECG])
for s in CANONICAL_SEEDS[1:]:
    train_unified_fusion('EXP0020', 'Block_F', 'ablation', 'MC0015', seed=s, epochs=25)
ds2_va_recs = [r for r in val_recs if r['DS']=='DS0002']
ds2_miss_ecg_idx = [i for i, r in enumerate(ds2_va_recs) if r['ModalityMask_CELS'][1] == 0]
if ds2_miss_ecg_idx:
    log_experiment('EXP0020', 'Block_F', 'ablation', 'MC0015', 'DS0002_Real_MISSING_ECG', y_ds2_all[ds2_miss_ecg_idx], p_ds2_all[ds2_miss_ecg_idx])
for m_idx, m_tag in [(1, 'ECG'), (2, 'Lab'), (3, 'Life')]:
    for p_drop in (0.10, 0.25, 0.50, 1.00):
        rng_s = np.random.default_rng(PRIMARY_SEED)
        def _stress(ds_id, m, pr=p_drop, col=m_idx):
            m2 = m.clone()
            drop_c = torch.from_numpy((rng_s.random(len(m2)) < pr)) & m2[:, col]
            m2[drop_c, col] = False
            return m2
        yv, pv, _ = eval_multicohort_model(champion_model, VA_COHORTS, override_mask=_stress)
        log_experiment('EXP0020', 'Block_F', 'ablation', 'MC0015', f'STRESS_drop{m_tag}_{int(p_drop*100)}pct', yv, pv)

In [ ]:
# Cell 7: Export Full S11 Validation Leaderboard, SHA-256 Checkpoint Manifest & Download Zip
df_res = pd.DataFrame(RESULTS_LOG)
df_res.to_csv(OUT_DIR / 's11_experiment_results.csv', index=False)

ckpt_manifest = []
for fpath in sorted(CKPT_DIR.glob('*')):
    if fpath.is_file():
        ckpt_manifest.append({'filename': fpath.name, 'bytes': fpath.stat().st_size, 'sha256': hashlib.sha256(fpath.read_bytes()).hexdigest()})
(OUT_DIR / 's11_checkpoint_manifest.json').write_text(json.dumps(ckpt_manifest, indent=2), encoding='utf-8')

# Create unified downloadable bundle for STEP-030 (Gate G10) & STEP-032 (Gate G11)
export_zip = OUT_DIR.parent / 's11_kaggle_export.zip'
with zipfile.ZipFile(export_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in OUT_DIR.rglob('*'):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUT_DIR)))

print(df_res[['EXP', 'Block', 'MC', 'Cohort', 'Seed', 'Val_ROCAUC_Raw', 'Val_PRAUC', 'Val_Brier_Cal', 'Val_ECE_Cal']].to_string(index=False))
print(f'\nSaved Stage S11 Export Archive: {export_zip} ({export_zip.stat().st_size/(1024*1024):.2f} MB) | Checkpoints={len(ckpt_manifest)}')